# Unidad 2. Preparación, limpieza y transformación de datos a escala

En este cuaderno vas a preparar una colección de mensajes en formato JSON para convertirla en una tabla revisable. La tarea forma parte de la simulación profesional de LexiData Observatorio: antes de analizar una colección textual, hay que importarla, estructurarla, revisar inconsistencias y documentar decisiones.

## Qué vas a practicar

1. Cargar un archivo JSON.
2. Inspeccionar su estructura.
3. Extraer campos anidados.
4. Crear un dataframe con Pandas.
5. Revisar duplicados y valores ausentes.
6. Crear columnas derivadas descriptivas.
7. Exportar un CSV preparado.

## 1. Preparación del entorno

In [ ]:
import json  
import re  
from pathlib import Path  

# Si falta pandas, activa el entorno virtual .venv de la asignatura e instala:
# python -m pip install -r 03_unidades/U2_preparacion_limpieza_transformacion/requirements_U2.txt
import pandas as pd

## 2. Ruta del archivo


In [ ]:
ruta_json = Path("U2_tweets_brutos.json")

with ruta_json.open(encoding="utf-8") as f:
    tweets = json.load(f) 

len(tweets), type(tweets), type(tweets[0]) 

## 3. Inspección inicial

In [ ]:
list(tweets[0].keys())

In [ ]:
tweets[0]["text"]

## 4. Extracción de campos y creación del dataframe

In [ ]:
def limpiar_fuente(html):
    return re.sub(r"<[^>]+>", "", html or "").strip() 


def extraer_fila(tweet):
    entidades = tweet.get("entities") or {} 
    usuario = tweet.get("user") or {} 
    texto = tweet.get("text") or ""
    return {
        "id_str": tweet.get("id_str"),
        "fecha_publicacion": tweet.get("created_at"),
        "texto": " ".join(texto.split()),
        "idioma": (tweet.get("metadata") or {}).get("iso_language_code") or tweet.get("lang"),
        "usuario": usuario.get("screen_name"),
        "ubicacion_usuario": usuario.get("location") or None,
        "fuente": limpiar_fuente(tweet.get("source")),
        "num_hashtags": len(entidades.get("hashtags") or []),
        "num_menciones": len(entidades.get("user_mentions") or []),
        "num_urls": len(entidades.get("urls") or []),
        "es_retuit": texto.startswith("RT @"),
        "longitud_texto": len(texto),
    }


df = pd.DataFrame([extraer_fila(t) for t in tweets])
df.head()

## 5. Revisión de inconsistencias

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df["id_str"].duplicated().sum()

In [ ]:
df["idioma"].value_counts(dropna=False)

## 6. Limpieza mínima y organización

In [ ]:
df_limpio = df.copy() 
df_limpio["ubicacion_usuario"] = df_limpio["ubicacion_usuario"].replace("", pd.NA).fillna("sin_ubicacion")
df_limpio = df_limpio.drop_duplicates(subset="id_str")
columnas = [
    "id_str", "fecha_publicacion", "idioma", "usuario", "ubicacion_usuario",
    "fuente", "texto", "longitud_texto", "num_hashtags", "num_menciones",
    "num_urls", "es_retuit"
]
df_limpio = df_limpio[columnas]
df_limpio.head()

## 7. Exportación

In [ ]:
df_limpio.to_csv("U2_tweets_preparados_alumno.csv", index=False, encoding="utf-8")
df_limpio.shape

## 8. Justificación final

Escribe aquí, con tus propias palabras, una justificación breve de las decisiones tomadas: qué columnas conservaste, qué inconsistencias revisaste, qué transformaciones aplicaste y por qué no realizaste interpretación del contenido textual.